# Capstone 5 — Expense tracker

The last capstone is a small ledger. Each entry has a date, a category, a description, and an amount. The ledger can filter a month, total by category, and compare a month with a budget. Amounts are stored as `Decimal`, because binary floats are the wrong type for money.

The demonstration loads a month of entries, prints the category totals, and flags the categories that crossed their budget.


In [1]:
from collections import defaultdict
from dataclasses import dataclass
from datetime import date
from decimal import Decimal

@dataclass(frozen=True)
class Expense:
    spent_on: date
    category: str
    description: str
    amount: Decimal

    def __post_init__(self):
        if self.amount <= 0:
            raise ValueError("amount must be positive")
        if not self.category.strip():
            raise ValueError("category is required")


class Ledger:
    def __init__(self):
        self._entries: list[Expense] = []

    def add(self, entry: Expense) -> None:
        self._entries.append(entry)

    def in_month(self, year: int, month: int) -> list[Expense]:
        return [e for e in self._entries if e.spent_on.year == year and e.spent_on.month == month]

    def totals(self, year: int, month: int) -> dict[str, Decimal]:
        grouped: dict[str, Decimal] = defaultdict(lambda: Decimal("0"))
        for entry in self.in_month(year, month):
            grouped[entry.category] += entry.amount
        return dict(sorted(grouped.items()))

    def alerts(self, year: int, month: int, budgets: dict[str, Decimal]) -> list[str]:
        messages = []
        spent = self.totals(year, month)
        for category, limit in sorted(budgets.items()):
            used = spent.get(category, Decimal("0"))
            if used > limit:
                messages.append(f"{category}: spent {used} against a budget of {limit}")
        return messages


ledger = Ledger()
rows = [
    (date(2026, 10, 2), "food", "lunch", "240"),
    (date(2026, 10, 3), "transport", "bus pass", "800"),
    (date(2026, 10, 6), "food", "groceries", "1450.50"),
    (date(2026, 10, 9), "books", "python reference", "699"),
    (date(2026, 10, 18), "food", "dinner", "560"),
    (date(2026, 9, 28), "food", "previous month", "100"),
]
for spent_on, category, description, amount in rows:
    ledger.add(Expense(spent_on, category, description, Decimal(amount)))

print("October totals")
for category, total in ledger.totals(2026, 10).items():
    print(f"  {category:<12} {total}")
print("month total:", sum(ledger.totals(2026, 10).values(), Decimal("0")))
budgets = {"food": Decimal("2000"), "transport": Decimal("700"), "books": Decimal("1000")}
print("alerts:")
for alert in ledger.alerts(2026, 10, budgets):
    print(" ", alert)


October totals
  books        699
  food         2250.50
  transport    800
month total: 3749.50
alerts:
  food: spent 2250.50 against a budget of 2000
  transport: spent 800 against a budget of 700


## What to notice

September's entry is stored and then excluded by `in_month`. The budget check reads the same totals as the report, so the alert cannot disagree with the printed table. `Decimal` is constructed from strings, not from binary floats, which keeps `1450.50` exact.

---

These five notebooks are independent. Run each from the top after a kernel restart.
